In [2]:
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
import torch
import os

/Users/isaac/.pyenv/versions/3.12.9/envs/jobable2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("ShashiVish/cover-letter-dataset")

In [4]:
dataset['train'].features

{'Job Title': Value('string'),
 'Preferred Qualifications': Value('string'),
 'Hiring Company': Value('string'),
 'Applicant Name': Value('string'),
 'Past Working Experience': Value('string'),
 'Current Working Experience': Value('string'),
 'Skillsets': Value('string'),
 'Qualifications': Value('string'),
 'Cover Letter': Value('string')}

In [5]:
def preprocess(row):
    X = f'''
    Job title: {row['Job Title']} ;
    Qualifications: {row['Preferred Qualifications']} ;
    Hiring Company: {row['Hiring Company']} ;
    Applicant Name: {row['Applicant Name']} ;
    Past Working Experience: {row['Past Working Experience']} ;
    Current Working Experience: {row['Current Working Experience']} ;
    Skillsets: {row['Skillsets']} ;
    Qualifications: {row['Qualifications']} ;
    '''

    y = row['Cover Letter']

    return {'input_text': X, 'target_text': y}

dataset = dataset.map(preprocess)

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Job Title', 'Preferred Qualifications', 'Hiring Company', 'Applicant Name', 'Past Working Experience', 'Current Working Experience', 'Skillsets', 'Qualifications', 'Cover Letter', 'input_text', 'target_text'],
        num_rows: 813
    })
    test: Dataset({
        features: ['Job Title', 'Preferred Qualifications', 'Hiring Company', 'Applicant Name', 'Past Working Experience', 'Current Working Experience', 'Skillsets', 'Qualifications', 'Cover Letter', 'input_text', 'target_text'],
        num_rows: 349
    })
})

In [7]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [8]:
def tokenize(row):
    model_inputs = tokenizer(
        row['input_text'],
        max_length=512,
        truncation=True,
        padding='max_length'
        )

    labels = tokenizer(
        row['target_text'],
        max_length=512,
        truncation=True,
        padding='max_length'
        )

    model_inputs['labels'] = labels['input_ids']

    return model_inputs

dataset = dataset.map(tokenize, batched=True)

In [9]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

In [10]:
os.environ["WANDB_DISABLED"] = "true"

In [11]:
# os.makedirs("./cover_letter_model", exist_ok=True)
training_args = TrainingArguments(
    output_dir="./cover_letter_model",
    eval_strategy="epoch",  # Updated deprecated argument
    save_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    report_to=["none"]  # Disabling Weights & Biases properly
)

In [4]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)


NameError: name 'Trainer' is not defined

In [3]:
trainer.train()


NameError: name 'trainer' is not defined

In [ ]:
model.save_pretrained("./cover_letter_model")
tokenizer.save_pretrained("./cover_letter_model")
